# Multi-Layer Perceptron from Scratch

In this notebook, we will construct a multi-layer perceptron (MLP) using `numpy` only. We consider the data sampled from the following model
$$ y = \sin x + 0.1 \epsilon, \quad \epsilon \sim \mathcal{N}(0, 1) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Data generation
np.random.seed(87)
N = 200                                             # number of data
X = np.linspace(-np.pi, np.pi, N).reshape(-1, 1)    # For the matrix operation. X: N x 1
y = np.sin(X) + 0.1 * np.random.randn(N, 1)         # For the matrix operation. Y: N x 1

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')

ax.set_title("Data")
ax.legend()

plt.show()

## 1. Linear regression
Obviously, fitting the given data with linear regression will give poor results.

> Even though we can obtain the closed-form solutions for linear regression, we use the gradient descent here.

In [ ]:
lr = 0.01
epochs = 5000         # number of iterations

w = np.zeros((1, 1))  # For the matrix operation
b = np.zeros(1)

lin_loss_history = np.zeros(epochs)

for epoch in range(epochs):
    y_pred_lin = X @ w + b
    error_lin = y_pred_lin - y

    lin_loss_history[epoch] = np.mean(error_lin**2)

    grad_w = 2 * X.T @ error_lin / len(X)
    grad_b = 2 * np.mean(error_lin)

    w -= lr * grad_w.T
    b -= lr * grad_b

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_lin, label="predicted", color='red')

ax.set_title("Linear Regression")
ax.legend()

plt.show()

In [ ]:
plt.plot(lin_loss_history)
plt.ylim([0, 0.6])
plt.xlabel("epochs")
plt.ylabel("MSE")
plt.title("Loss history (Linear regression)")
plt.show()

## 2. Non-linearity
### Activation functions

**ReLU**
$$ \text{ReLU}(x) = \max (0, x) = \begin{cases}
x, & x > 0, \\
0, & x \le 0.
\end{cases} $$
Even though this function is not differentiable at $x = 0$, we define
$$ \text{ReLU}'(x) = \begin{cases}
1, & x > 0, \\
0, & x \le 0.
\end{cases} $$

**tanh**
$$ \tanh (x) = \frac{e^x -e^{-x}}{e^x + e^{-x}}. $$
It's derivative is given by
$$ \tanh'(x) = 1 - \tanh^2(x). $$

In [ ]:
# ReLU
def relu(x):
    return np.maximum(0, x)
def relu_grad(x):
    return (x > 0).astype(float)

# tanh
def tanh(z):
    return np.tanh(z)

def tanh_grad(z):
    return 1 - np.tanh(z) ** 2

In [ ]:
xx = np.linspace(-5, 5, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(xx, relu(xx), color='red')
axes[0].set_title("ReLU")
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].axvline(0, color='gray', lw=0.8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(xx, tanh(xx), color='blue')
axes[1].set_title("tanh")
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].axvline(0, color='gray', lw=0.8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.1 Simplest neural network (1 hidden layer with `hidden_dim` = 1)
We want to include 1 hidden layer of dimension 1 between input layer and output layer.

In [ ]:
# Hyperparmeters
input_dim = 1
hidden_dim = 1
output_dim = 1
lr = 0.01
epochs = 5000


# Initialization
np.random.seed(87)
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

In [ ]:
## ReLU

loss_history_relu = []

for epoch in range(epochs):
    # Forward
    h1 = X @ W1 + b1
    z1 = relu(h1) 
    y_pred_relu = z1 @ W2 + b2

    # Loss (MSE)
    error_relu = y_pred_relu - y
    loss_relu = np.mean(error_relu**2)
    loss_history_relu.append(loss_relu)

    # Backward
    dy = 2 * error_relu / len(X)
    
    dW2 = z1.T @ dy
    db2 = np.sum(dy, axis=0, keepdims=True)

    dz1 = dy @ W2.T
    dh1 = dz1 * relu_grad(h1)
    dW1 = X.T @ dh1
    db1 = np.sum(dh1, axis=0, keepdims=True)

    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: MSE = {loss_relu:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_relu, label="predicted", color='red')

ax.set_title(f"ReLU (after {epochs} epochs)")
ax.legend()

plt.show()

In [ ]:
plt.plot(loss_history_relu)
plt.ylim([0, 0.6])
plt.xlabel("epochs")
plt.ylabel("MSE")
plt.title("Loss history (ReLU)")
plt.show()

In [ ]:
# Hyperparmeters
input_dim = 1
hidden_dim = 1
output_dim = 1
lr = 0.01
epochs = 5000


# Initialization
np.random.seed(87)
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

In [ ]:
## tanh

loss_history_tanh = []

for epoch in range(epochs):
    # Forward
    h1 = X @ W1 + b1
    z1 = tanh(h1)
    y_pred_tanh = z1 @ W2 + b2

    # Loss (MSE)
    error_tanh = y_pred_tanh - y
    loss_tanh = np.mean(error_tanh**2)
    loss_history_tanh.append(loss_tanh)

    # Backward
    dy = 2 * error_tanh / len(X)
    
    dW2 = z1.T @ dy
    db2 = np.sum(dy, axis=0, keepdims=True)

    dz1 = dy @ W2.T
    dh1 = dz1 * tanh_grad(h1)
    dW1 = X.T @ dh1
    db1 = np.sum(dh1, axis=0, keepdims=True)

    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: MSE = {loss_tanh:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_tanh, label="predicted", color='red')

ax.set_title(f"tanh (after {epochs} epochs)")
ax.legend()

plt.show()

In [ ]:
plt.plot(loss_history_tanh)
plt.ylim([0, 0.6])
plt.xlabel("epochs")
plt.ylabel("MSE")
plt.title("Loss history (tanh)")
plt.show()

### 2.2 What if we use the linear activation function?

In [ ]:
# Identity (linear) activation:  g(x) = x,  g'(x) = 1
def identity(x):
    return x
def identity_deriv(x):
    return np.ones_like(x)

In [ ]:
## identity

# Hyperparmeters
input_dim = 1
hidden_dim = 1
output_dim = 1
lr = 0.01
epochs = 5000

# Initialization
np.random.seed(87)
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

loss_history_id = []

for epoch in range(epochs):
    # Forward
    h1 = X @ W1 + b1
    z1 = identity(h1)               # <-- only change vs. ReLU: activation is g(x) = x
    y_pred_id = z1 @ W2 + b2

    # Loss (MSE)
    error_id = y_pred_id - y
    loss_id = np.mean(error_id**2)
    loss_history_id.append(loss_id)

    # Backward
    dy = 2 * error_id / len(X)

    dW2 = z1.T @ dy
    db2 = np.sum(dy, axis=0, keepdims=True)

    dz1 = dy @ W2.T
    dh1 = dz1 * identity_deriv(h1)  # <-- derivative of identity is 1
    dW1 = X.T @ dh1
    db1 = np.sum(dh1, axis=0, keepdims=True)

    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: MSE = {loss_id:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_id, label="predicted", color='red')

ax.set_title(f"1 Hidden Layer (after {epochs} epochs)")
ax.legend()

plt.show()

In [ ]:
plt.plot(loss_history_id)
plt.ylim([0, 0.6])
plt.xlabel("epochs")
plt.ylabel("MSE")
plt.title("Loss history (1 Hidden Layer)")
plt.show()

In [ ]:
# Collapse the two-layer identity network into its effective slope & intercept
slope = (W1 @ W2).item()          # W1 * W2
intercept = (b1 @ W2 + b2).item() # b1 * W2 + b2

print("Identity MLP  : slope = %.5f, intercept = %.5f" % (slope, intercept))
print("Linear reg    : slope = %.5f, intercept = %.5f" % (w.item(), b.item()))
print("Max |prediction difference| = %.2e" % np.max(np.abs(y_pred_id - y_pred_lin)))

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_lin, label="linear regression", color='red', linewidth=3, alpha=0.5)
ax.plot(X, y_pred_id, label="identity-activation MLP", color='blue', linestyle='--')

ax.set_title("Identity activation reproduces linear regression")
ax.legend()

plt.show()

### 2.3 Go Deeper and Wider

Now, we consider a deeper (2 hidden layers) and wider (`hidden_dim` = 50) network.

In [ ]:
## ReLU MLP

# Hyperparmeters
input_dim = 1
h1_dim = 50
h2_dim = 50
output_dim = 1
lr = 0.01
epochs = 5000

# Initialization
W1 = np.random.randn(input_dim, h1_dim) * 0.1
b1 = np.zeros((1, h1_dim))

W2 = np.random.randn(h1_dim, h2_dim) * 0.1
b2 = np.zeros((1, h2_dim))

W3 = np.random.randn(h2_dim, output_dim) * 0.1
b3 = np.zeros((1, output_dim))

loss_history_mlp = []

for epoch in range(epochs):
    # Forward
    h1 = X @ W1 + b1
    z1 = relu(h1)

    h2 = z1 @ W2 + b2
    z2 = relu(h2)

    y_pred_mlp = z2 @ W3 + b3

    # Loss
    error_mlp = y_pred_mlp - y
    loss_mlp = np.mean(error_mlp ** 2)
    loss_history_mlp.append(loss_mlp)

    # Backward
    dy = 2 * error_mlp / len(X)

    dW3 = z2.T @ dy
    db3 = np.sum(dy, axis=0)

    dz2 = dy @ W3.T
    dh2 = dz2 * relu_grad(h2)

    dW2 = z1.T @ dh2
    db2 = np.sum(dh2, axis=0)

    dz1 = dh2 @ W2.T
    dh1 = dz1 * relu_grad(h1)

    dW1 = X.T @ dh1
    db1 = np.sum(dh1, axis=0)

    # Update
    W3 -= lr * dW3
    b3 -= lr * db3
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: MSE = {loss_mlp:.4f}")

In [ ]:
# ## tanh MLP

# # Hyperparmeters
# input_dim = 1
# h1_dim = 50
# h2_dim = 50
# output_dim = 1
# lr = 0.01
# epochs = 5000

# # Initialization
# W1 = np.random.randn(input_dim, h1_dim) * 0.1
# b1 = np.zeros((1, h1_dim))

# W2 = np.random.randn(h1_dim, h2_dim) * 0.1
# b2 = np.zeros((1, h2_dim))

# W3 = np.random.randn(h2_dim, output_dim) * 0.1
# b3 = np.zeros((1, output_dim))

# loss_history_mlp = []

# for epoch in range(epochs):
#     # Forward
#     h1 = X @ W1 + b1
#     z1 = tanh(h1)

#     h2 = z1 @ W2 + b2
#     z2 = tanh(h2)

#     y_pred_mlp = z2 @ W3 + b3

#     # Loss
#     error_mlp = y_pred_mlp - y
#     loss_mlp = np.mean(error_mlp ** 2)
#     loss_history_mlp.append(loss_mlp)

#     # Backward
#     dy = 2 * error_mlp / len(X)

#     dW3 = z2.T @ dy
#     db3 = np.sum(dy, axis=0)

#     dz2 = dy @ W3.T
#     dh2 = dz2 * tanh_grad(h2)

#     dW2 = z1.T @ dh2
#     db2 = np.sum(dh2, axis=0)

#     dz1 = dh2 @ W2.T
#     dh1 = dz1 * tanh_grad(h1)

#     dW1 = X.T @ dh1
#     db1 = np.sum(dh1, axis=0)

#     # Update
#     W3 -= lr * dW3
#     b3 -= lr * db3
#     W2 -= lr * dW2
#     b2 -= lr * db2
#     W1 -= lr * dW1
#     b1 -= lr * db1

#     if epoch % 100 == 0:
#         print(f"Epoch {epoch}: MSE = {loss_mlp:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(X, y, s=5, label="data", color='black')
ax.plot(X, y_pred_mlp, label="predicted", color='red')

ax.set_title(f"MLP (after {epochs} epochs)")
ax.legend()

plt.show()

In [ ]:
plt.plot(loss_history_mlp)
plt.ylim([0, 0.6])
plt.xlabel("epochs")
plt.ylabel("MSE")
plt.title("Loss history (MLP)")
plt.show()

#### Discussion
In this example, the MLP was trained using the entire dataset. As a result, the loss continues to decrease with more training epochs—but this does not guarantee better performance on new, unseen data. In fact, it may lead to **overfitting**.

To achieve a better generalization, how could you redesign the training process?

- How would you split the dataset?
- What would be an appropriate stopping condition to prevent overfitting?